# MD5 消息摘要算法第五版

**摘要：** MD5是128位哈希函数，曾广泛应用，因碰撞漏洞已被淘汰仅非安全场景使用

- **类别：** 现代密码学
- **难度：** 中等
- **预计用时：** 2.5小时
- **先修：** 基础哈希概念、二进制运算、模运算、布尔函数
- **学习目标：** 掌握MD5原理与实现

> 说明：本课程强调“可运行 + 可解释 + 可练习”。代码尽量仅使用 Python 标准库（Pyodide 友好）。

## 你将获得什么

- **固定输出 128位：** 输出长度固定为128比特
- **64轮迭代 非线性：** 64轮迭代含四种布尔函数
- **已被攻破 无安全性：** 存在快速碰撞攻击漏洞

## 学习路线图（从直觉到实现）

我们把学习过程拆成 4 层，每一层都尽量给出“可验证”的产物：

1. **直觉层**：能说清楚它解决什么安全目标，以及为什么需要“密钥/参数/随机性”。
2. **符号层**：能把关键变换写成一个简短公式，例如 $y=f(x,k)$ 或 $$y=f(x,k)\bmod n$$。
3. **实现层**：能写出可运行的函数/类，并通过至少 3 条 `assert` 自测。
4. **对抗层**：能指出它可能被怎么攻（至少一个思路），并用代码做一个最小实验验证。

> 你会发现：能通过“对抗层”的小实验，往往才算真正理解。


## 应用场景与安全性讨论（扩充阅读）

这一主题通常会在以下场景出现（不同主题侧重点不同）：

- **教学/入门**：用简化模型理解“密钥 + 变换”的思想。
- **工程系统**：用成熟算法与标准协议实现保密性/完整性/认证。
- **攻防分析**：通过已知攻击理解“为什么某些方案不再推荐”。

### 安全目标（Security Goals）

常见目标包括：

- **保密性（Confidentiality）**：未授权者无法获得明文信息。
- **完整性（Integrity）**：消息在传输中被篡改能被发现。
- **认证（Authentication）**：确认对方是谁（或确认数据来自谁）。
- **不可否认（Non-repudiation）**：事后不能否认曾经生成/发送过某信息（通常与签名相关）。

### 威胁模型（Threat Model）

做任何“安全性结论”之前，先明确攻击者能做什么：

- 只能看到密文？还是还能选择明文（chosen-plaintext）？
- 能否篡改、重放、插入消息？
- 能否获取部分密钥/随机数？是否存在侧信道？

> 调试提示：如果你觉得“算法没问题但结果不对”，先检查你实现的威胁模型是否一致——例如你是否在复用 nonce/IV，或者把编码/填充当成了算法的一部分。


## 环境准备

In [ ]:
# 课程通用导入（尽量只用标准库）
import math
import random
import string
import secrets
import hashlib
import hmac
import itertools
import statistics
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional, Iterable

random.seed(0)  # 为了让演示更可复现


## 热身：数据与表示

很多实现问题来自“数据表示”而不是算法本身。请确保你能区分：

- 文本：`str`
- 字节：`bytes`
- 整数：`int`

并能相互转换。

In [ ]:
msg = "Hello, 密码学"
b = msg.encode("utf-8")
print(type(msg), type(b))  # 预期输出：<class 'str'> <class 'bytes'>
print(b[:10])              # 预期输出：前10个字节（与编码有关）
print(b.hex()[:20])        # 预期输出：十六进制字符串前20个字符


### 工具函数：字节/整数/十六进制

In [ ]:
def bytes_to_hex(b: bytes) -> str:
    return b.hex()

def hex_to_bytes(h: str) -> bytes:
    h = h.strip().lower()
    if h.startswith("0x"):
        h = h[2:]
    return bytes.fromhex(h)

def int_to_bytes(x: int, length: int, byteorder: str = "big") -> bytes:
    return x.to_bytes(length, byteorder=byteorder, signed=False)

def bytes_to_int(b: bytes, byteorder: str = "big") -> int:
    return int.from_bytes(b, byteorder=byteorder, signed=False)

assert hex_to_bytes(bytes_to_hex(b"ABC")) == b"ABC"


## 核心内容分步讲解

### Step 1: 了解MD5的起源与背景

1. MD5全称为Message Digest Algorithm 5（消息摘要算法第五版），由RSA三位发明人之一的Ron Rivest在1991年设计并发布。
2. 它是MD系列哈希函数的最后一代，演进路径为MD2（1989，针对8位机优化）→ MD4（1990，速度快但安全性不足）→ MD5（1991，改进MD4安全弱点）。
3. 设计背景源于90年代初互联网发展对数字签名、文件完整性校验、消息防篡改的需求，核心是在MD4基础上加强运算步骤以提升安全性。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 2: 梳理MD5的应用与弃用历程

1. 标准化与应用阶段：MD5快速成为事实上的行业标准，广泛应用于早期SSL/TLS（HTTPS协议）、PGP/SSH加密工具、RSA/DSA数字签名哈希预处理、md5sum文件校验工具等场景。
2. 弱点暴露阶段：1996年Hans Dobbertin指出其压缩函数存在弱点；2004年王小云团队公布实用碰撞攻击方法；2008年研究人员利用MD5碰撞伪造CA证书，直接威胁互联网安全。
3. 弃用阶段：2008年起IETF、NIST等机构明确禁用MD5，现代应用全面迁移至SHA-2或SHA-3，目前仅用于非安全场景（如快速文件校验）。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 3: 掌握MD5的总体算法结构

MD5采用与SHA-1同源的Merkle–Damgård经典哈希结构，整体执行流程分为四个核心环节：
1. 消息填充（Padding）：对输入消息进行标准化填充，保证长度符合算法要求；
2. 消息分块：将填充后的消息拆分为固定大小的512比特分组；
3. 迭代压缩函数：对每个分组执行64轮非线性迭代运算，更新哈希状态；
4. 输出摘要：所有分组处理完成后，拼接最终哈希状态得到128位消息摘要。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 4: 执行消息预处理操作

预处理是MD5保证输入唯一性和算法规范性的关键步骤，分为填充和分块两步：
1. 填充规则：
   - 设原始消息的比特长度为$L$，先在消息末尾添加1个比特的1；
   - 补充若干个0比特，直到消息总长度 ≡ 448 (mod 512)；
   - 最后附加$L$的64位二进制表示，使填充后总长度为512的整数倍。
2. 分块操作：将填充后的完整消息按512比特为单位拆分，得到若干分组，记为$M^{(1)}, M^{(2)}, …, M^{(N)}$。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 5: 初始化MD5哈希状态（IV）

MD5的内部哈希状态由4个32位寄存器组成，算法预置固定初始值：
$A = 0x67452301$
$B = 0xEFCDAB89$
$C = 0x98BADCFE$
$D = 0x10325476$
这四个寄存器的初始值是MD5算法的核心常量，作为迭代压缩计算的起始状态，最终的128位MD5摘要由这四个寄存器的最终值拼接而成，即$MD5(M) = A∥B∥C∥D$（$∥$表示比特串拼接）。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 6: 理解核心压缩函数的构成

MD5的压缩函数是算法核心，处理每个512比特消息分组，分为三个关键部分：
1. 消息分组拆分：将512比特分组拆分为16个32位字，记为$M_0, M_1, …, M_{15}$；
2. 非线性布尔函数：64轮迭代分为4组（各16轮），分别使用F、G、H、I四种布尔函数：
   - 0-15轮：$F(X,Y,Z)=(X∧Y)∨(¬X∧Z)$
   - 16-31轮：$G(X,Y,Z)=(X∧Z)∨(Y∧¬Z)$
   - 32-47轮：$H(X,Y,Z)=X⊕Y⊕Z$
   - 48-63轮：$I(X,Y,Z)=Y⊕(X∨¬Z)$
3. 轮常数与移位量：每轮使用的常数$T_i=⌊2^{32}⋅|sin(i+1)|⌋$，且每16轮使用一组固定的循环左移位数$s$。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 7: 执行64轮迭代运算

对每个消息分组执行64轮迭代运算，核心步骤如下：
1. 初始化本轮工作寄存器：将当前全局哈希状态$(A,B,C,D)$赋值为本轮工作寄存器；
2. 逐轮计算：
   - 根据轮次选择对应的布尔函数$f$、消息字$M_k$、常数$T_i$和移位量$s$；
   - 执行核心迭代公式：$A = B + ROL_s(A + f(B,C,D) + M_k + T_i)$（$ROL_s$为循环左移$s$位）；
   - 循环更新寄存器：$(A,B,C,D) → (D, 新B, B, C)$；
3. 状态更新：64轮运算完成后，将本轮得到的新$(A,B,C,D)$与原始全局哈希状态做模$2^{32}$加法，更新全局哈希状态：
   $A = A + A_{old}, B = B + B_{old}, C = C + C_{old}, D = D + D_{old}$。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 8: 生成最终MD5哈希输出

当所有512比特消息分组处理完成后，将最终的4个32位寄存器值按$A∥B∥C∥D$的顺序拼接，得到长度为128位（16字节）的MD5消息摘要。
在实际应用中，通常将128位二进制摘要转换为32位十六进制字符串展示，这也是md5sum等工具的标准输出形式。
需要注意的是，MD5在输出时采用小端序（Little Endian）存储，即寄存器的字节顺序会反转后再转换为十六进制。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 9: 实现并验证MD5算法

基于MD5的数学原理，可通过Python完整实现算法并验证正确性：
1. 实现核心工具函数：如32位整数循环左移函数left_rotate；
2. 消息预处理：完成比特填充（添加1比特、补0、附加长度）和分块；
3. 定义算法常量：包括初始哈希值、64轮常数k、64轮移位量s；
4. 分块处理：对每个512比特块执行64轮迭代运算，更新全局哈希状态；
5. 结果输出：将最终哈希状态转换为小端序的十六进制字符串；
6. 验证：使用标准测试用例（空字符串、"abc"、"message digest"等）验证实现结果与已知MD5值一致。


> 提示：注意MD5的小端序处理规则，这是与SHA-1的重要区别之一

**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
import struct


def left_rotate(value, amount):
    """32位整数的循环左移"""
    return ((value << amount) | (value >> (32 - amount))) & 0xFFFFFFFF


def md5(message):
    """
    计算输入消息的MD5散列值
    
    参数:
        message (str 或 bytes): 要计算散列值的消息
    
    返回:
        str: MD5散列值的十六进制字符串表示
    """
    # 将输入消息转换为字节
    if isinstance(message, str):
        message = message.encode('utf-8')
    
    # MD5算法使用的初始哈希值
    h = [0x67452301, 0xEFCDAB89, 0x98BADCFE, 0x10325476]
    
    # 预处理：添加填充
    message_len = len(message)
    message += b'\x80'  # 添加一个1位，其余为0
    
    # 填充到长度为448 mod 512
    while len(message) % 64 != 56:
        message += b'\x00'
    
    # 添加原始消息长度（以位为单位）
    message += struct.pack('<Q', message_len * 8)
    
    # MD5中使用的常数表（sin函数的绝对值）
    k = [
        0xd76aa478, 0xe8c7b756, 0x242070db, 0xc1bdceee,
        0xf57c0faf, 0x4787c62a, 0xa8304613, 0xfd469501,
        0x698098d8, 0x8b44f7af, 0xffff5bb1, 0x895cd7be,
        0x6b901122, 0xfd987193, 0xa679438e, 0x49b40821,
        0xf61e2562, 0xc040b340, 0x265e5a51, 0xe9b6c7aa,
        0xd62f105d, 0x02441453, 0xd8a1e681, 0xe7d3fbc8,
        0x21e1cde6, 0xc33707d6, 0xf4d50d87, 0x455a14ed,
        0xa9e3e905, 0xfcefa3f8, 0x676f02d9, 0x8d2a4c8a,
        0xfffa3942, 0x8771f681, 0x6d9d6122, 0xfde5380c,
        0xa4beea44, 0x4bdecfa9, 0xf6bb4b60, 0xbebfbc70,
        0x289b7ec6, 0xeaa127fa, 0xd4ef3085, 0x04881d05,
        0xd9d4d039, 0xe6db99e5, 0x1fa27cf8, 0xc4ac5665,
        0xf4292244, 0x432aff97, 0xab9423a7, 0xfc93a039,
        0x655b59c3, 0x8f0ccc92, 0xffeff47d, 0x85845dd1,
        0x6fa87e4f, 0xfe2ce6e0, 0xa3014314, 0x4e0811a1,
        0xf7537e82, 0xbd3af235, 0x2ad7d2bb, 0xeb86d391
    ]
    
    # 每轮的左移位数
    s = [
        7, 12, 17, 22, 7, 12, 17, 22, 7, 12, 17, 22, 7, 12, 17, 22,
        5,  9, 14, 20, 5,  9, 14, 20, 5,  9, 14, 20, 5,  9, 14, 20,
        4, 11, 16, 23, 4, 11, 16, 23, 4, 11, 16, 23, 4, 11, 16, 23,
        6, 10, 15, 21, 6, 10, 15, 21, 6, 10, 15, 21, 6, 10, 15, 21
    ]
    
    # 处理每个512位的块
    for chunk_start in range(0, len(message), 64):
        chunk = message[chunk_start:chunk_start + 64]
        
        # 将块分解为16个32位的小端序字
        w = list(struct.unpack('<16I', chunk))
        
        # 初始化这一轮的哈希值
        a, b, c, d = h
        
        # 主循环
        for i in range(64):
            if i < 16:
                f = (b & c) | (~b & d)
                g = i
            elif i < 32:
                f = (d & b) | (~d & c)
                g = (5 * i + 1) % 16
            elif i < 48:
                f = b ^ c ^ d
                g = (3 * i + 5) % 16
            else:
                f = c ^ (b | ~d)
                g = (7 * i) % 16
            
            f = (f + a + k[i] + w[g]) & 0xFFFFFFFF
            a, b, c, d = d, (b + left_rotate(f, s[i])) & 0xFFFFFFFF, b, c
        
        # 添加这一轮的哈希值到结果中
        h[0] = (h[0] + a) & 0xFFFFFFFF
        h[1] = (h[1] + b) & 0xFFFFFFFF
        h[2] = (h[2] + c) & 0xFFFFFFFF
        h[3] = (h[3] + d) & 0xFFFFFFFF
    
    # 产生最终的哈希值（小端序）
    result = struct.pack('<4I', *h)
    return result.hex()

# 演示验证
test_cases = [
    "",
    "a",
    "abc",
    "message digest",
    "abcdefghijklmnopqrstuvwxyz",
    "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789",
    "1234567890" * 8
]

for message in test_cases:
    hash_value = md5(message)
    print(f"MD5('{message}') = {hash_value}")

# 预期输出：根据输入得到对应结果（请与讲解示例对照）

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

## 综合示例：端到端流程 + 自测

In [ ]:
# 端到端模板：将主题核心操作封装成接口
# 你可以把 encrypt/decrypt 或 hash/sign/verify 等组合在一起

def pipeline(data: bytes) -> bytes:
    # TODO: 替换为你的端到端流程
    return hashlib.sha256(data).digest()

def check_pipeline():
    a = pipeline(b"hello")
    b = pipeline(b"hello")
    c = pipeline(b"hellp")
    assert a == b
    assert a != c
    print("端到端自测通过")  # 预期输出：端到端自测通过

check_pipeline()


## 小实验：敏感性（雪崩效应）

In [ ]:
def hamming_distance_bytes(a: bytes, b: bytes) -> int:
    if len(a) != len(b):
        raise ValueError("长度必须相同才能计算差异度")
    return sum(x != y for x, y in zip(a, b))

def flip_one_bit(data: bytes, bit_index: int = 0) -> bytes:
    if not data:
        return data
    byte_i = bit_index // 8
    bit_i = bit_index % 8
    byte_i = byte_i % len(data)
    mask = 1 << bit_i
    arr = bytearray(data)
    arr[byte_i] ^= mask
    return bytes(arr)

def core_transform(data: bytes) -> bytes:
    # TODO: 替换成你的核心变换
    # 示例：用 SHA-256 作为“对照组”
    return hashlib.sha256(data).digest()

base = b"hello world"
out1 = core_transform(base)
out2 = core_transform(flip_one_bit(base, 0))

print("输出长度(字节):", len(out1))  # 预期输出：32（若使用SHA-256）
print("差异度(字节):", hamming_distance_bytes(out1, out2))  # 预期输出：通常 > 10


## 附录A：最常用的数学与位运算背景

### 1. 模运算（Modular Arithmetic）

当我们写 $$a \equiv b \pmod n$$ 时，意思是 $a$ 与 $b$ 除以 $n$ 的余数相同，也就是 $n$ 整除 $a-b$。

常见等价写法：

- $a \bmod n$：把 $a$ 映射到 $0..n-1$ 的代表元
- 若 $a<0$，Python 仍保证 $a \bmod n \in [0,n-1]$

**一个很重要的直觉：** 模运算会“折叠”整数轴，把无限多的整数映射到有限集合，所以很多密码体制会用到它来构造“封闭”的运算空间。

### 2. 最大公因数与互质

若 $$\gcd(a,n)=1$$，我们称 $a$ 与 $n$ 互质（coprime）。互质非常重要，因为它意味着 $a$ 在模 $n$ 意义下存在乘法逆元 $a^{-1}$，满足：

$$a\cdot a^{-1} \equiv 1 \pmod n$$

### 3. 扩展欧几里得算法（Extended Euclid）

它不仅能算 $\gcd(a,b)$，还能找到整数 $x,y$ 使得：

$$ax+by=\gcd(a,b)$$

当 $\gcd(a,n)=1$ 时，$x$ 就是 $a$ 关于模 $n$ 的逆元（可能为负，需要再取模）。

### 4. 异或（XOR）

在字节层面，异或常写成 $c=a\oplus b$，它有几个极其重要的性质：

- $a\oplus a=0$
- $a\oplus 0=a$
- $(a\oplus b)\oplus b=a$（可逆性）

因此很多流密码/分组模式会用异或把“密钥流”与明文混合。

> 调试提示：如果你发现解密结果不等于明文，先检查“同一份密钥流是否被一致地使用”，以及字节对齐是否正确。


In [ ]:
# 附录A配套代码：gcd、逆元、异或操作的最小实现与自测

def egcd(a: int, b: int) -> Tuple[int, int, int]:
    """返回 (g, x, y) 使得 a*x + b*y = g = gcd(a,b)"""
    if b == 0:
        return (a, 1, 0)
    g, x1, y1 = egcd(b, a % b)
    x = y1
    y = x1 - (a // b) * y1
    return (g, x, y)

def modinv(a: int, n: int) -> int:
    g, x, _ = egcd(a, n)
    if g != 1:
        raise ValueError("不存在逆元：a 与 n 不互质")
    return x % n

print(math.gcd(3, 26))     # 预期输出：1
print(modinv(3, 26))       # 预期输出：9，因为 3*9=27≡1(mod26)

def xor_bytes(a: bytes, b: bytes) -> bytes:
    if len(a) != len(b):
        raise ValueError("xor 需要等长字节串")
    return bytes(x ^ y for x, y in zip(a, b))

p = b"ABC"
k = b"\x01\x01\x01"
c = xor_bytes(p, k)
p2 = xor_bytes(c, k)
print(c)   # 预期输出：b'@BA'（因为 0x41^1=0x40 等）
print(p2)  # 预期输出：b'ABC'


## 常见错误 / 踩坑点 / 调试提示

> 1. **编码问题**：`str`/`bytes` 混用导致哈希/加解密结果不一致。
>
> 2. **参数合法性**：例如需要互质、需要固定长度、需要随机数不可复用。
>
> 3. **端序与位序**：大端/小端混淆、位操作方向写反。
>
> 4. **测试不足**：缺少边界与异常路径测试。
>
> 5. **把演示当安全方案**：课程中的简化实现不等价于生产安全实现。

## 练习题（含更详细要求）

### 练习1：功能完善（必做）
- 目标：把你的核心函数做成“更好用”的版本  
- 要求：
  - 输入支持 `str` 与 `bytes`
  - 对非法参数给出清晰报错（`ValueError`）
  - 至少写 5 条 `assert`（覆盖正常/边界/异常）

### 练习2：最小攻防实验（推荐）
- 目标：实现一个最小的攻击/对抗脚本  
- 示例方向（任选其一）：
  - 暴力枚举 key space
  - 频率分析/统计偏差
  - 重放/篡改检测失败示例
  - 碰撞/相同摘要的构造（仅演示，不鼓励用于真实攻击）

### 练习3：写一段“工程化清单”（可选）
- 目标：把课程知识迁移到真实系统  
- 建议包含：
  - 参数选择（key 长度、随机数、模式）
  - 依赖库与实现来源（优先标准/权威实现）
  - 安全审计点（日志、异常、边界）


In [ ]:
# 练习参考答案框架（示例）
# 说明：这里只提供“结构”，你需要填入你自己的实现。

def solve_ex1(data):
    if isinstance(data, str):
        data_b = data.encode("utf-8")
    elif isinstance(data, (bytes, bytearray)):
        data_b = bytes(data)
    else:
        raise ValueError("只支持 str/bytes")
    return hashlib.sha256(data_b).hexdigest()

assert solve_ex1("a") == solve_ex1("a")
assert solve_ex1("a") != solve_ex1("b")
try:
    solve_ex1(123)
except ValueError:
    pass

print("练习1框架可运行")  # 预期输出：练习1框架可运行


## 小结

把今天的内容压缩成 3 句话：

1. 这个主题的核心变换是 $y=f(x,k)$（或 $$y=f(x,k)\bmod n$$）。
2. 正确实现需要重视：数据表示、参数合法性、测试向量与边界条件。
3. 真正理解来自：能写出最小攻防实验，并解释现象原因。


## 随堂小测（10题）
1. 这个主题的“密钥/参数”是什么？它决定了哪些行为？
2. 为什么需要模运算/置换/异或等操作？分别带来什么效果？
3. 在你的实现里，哪一步最容易写错？你如何用测试发现它？
4. 若攻击者拥有密文（或摘要），最直接的攻击方式是什么？
5. 在什么场景下“能解密”不等于“安全”？举一个例子。
6. 是否存在“重复使用随机数/nonce/IV”的风险？为什么危险？
7. 如果输入非常长，你的实现是否会变慢？瓶颈在哪？
8. 你能写出一个最小“对照实验”，让输出变化更直观吗？
9. 如果要用于真实系统，你会替换/增强哪一部分？
10. 用一句话总结：你学到的最重要概念是什么？

> 自评：能回答 7/10 且能跑通练习，就算达标。
